# Einführung in die Grundlagen der natürlichen Sprachverarbeitung mit Python

Dieses Notebook ist im Rahmen der Fortbildungsreihe "Panorama Text: Maschinelle Erfassung, Verarbeitung und Kontextualisierung von geschriebener Sprache für die Geisteswissenschaften" und konkret des Workshops [Einführung in die Grundlagen der natürlichen Sprachverarbeitung mit Python](https://www.it.fu-berlin.de/unsere-services/kompetenzentwicklung/fortbildungen/workshops/E-Research/2026-04-20-Python.html) im Sommersemester 2026 an der FU Berlin entstanden. [Zusammenfassung Inhalte]

Beitragende und ihre Rollen anhand der [Contributor Role Taxonomy (CRediT) taxonomy](https://doi.org/10.1002/leap.1210):
* Sophie Schneider ([@BibWiss](https://github.com/BibWiss)): Conceptualization, Software, Writing - Original Draft

## Installation, Import von Bibliotheken

Zum Ausführen des Notebooks wird eine Installation von Python benötigt, dafür haben wir miniconda installiert und mithilfe des Paketmanagers conda ein virtual environment erstellt, in dem Python --version ... und jupyter notebook laufen. Jupyter Notebook ist eine sogenannte Python-Bibliothek (auch Paket, Def?), wir werden allerdings auch noch auf Funktionalitäten anderer Bibliotheken zugreifen.

Hierfür müssen wir jede Bibliothek (die keine Python Standard Library ist) einmalig installieren und dann nochmal zusätzlich importieren. Python-Kommentare beginnen mit `#` oder stehen zwischen `"""<auskommentierter Python-Code>"""` (mehrzeiliger Kommentar).

💡 **Aufgabe**: Entferne in der zweiten Codezelle die `#`, installiere alle weiteren benötigten Pakete und kommentiere den Code dann mit einem mehrzeiligen Kommentar aus.

In [19]:
# ! conda env list #zeigt mit * an, welche env aktiviert ist

In [ ]:
# ! conda install 

In [6]:
import re

## 1. Data Aggregation

Um die Komplexität möglich gering zu halten, arbeiten wir erstmal mit kleinen Textschnipseln. Die folgenden stammen aus der Wikipedia (genauere Angaben s.u.) und wurden mithilfe eines [Zufallsgenerators](https://ebildungslabor.github.io/zufallswikipedia/) aufgefunden. Um auf die Texte zugreifen zu können, müssen wir jeweils eine zugehörige Variable definieren, nach dem Schema `<variable> = <wert>`.

💡 **Aufgabe**: Definiere eine Variable `text3` und ordne ihr einen selbst gewählten Wert (kurzen Textabschnitt) zu. 

In [1]:
# "Mets (Athen)" via Wikipedia: https://de.wikipedia.org/w/index.php?title=Mets_(Athen)&oldid=263876565, CC BY-SA 4.0.
text1 = "Mets (griechisch Μετς) ist ein Stadtteil der griechischen Hauptstadt Athen. Im Viertel befanden sich früher zahlreiche Stadtvillen, von denen einige im nördlichen Teil bis heute erhalten sind. Auf dem Hügel Logginos wurde inzwischen ein Erholungspark angelegt. Mets verdankt seinen Namen einer Bierbrauerei, die vom bayerischen Brauer Karl Fuchs eröffnet wurde, derselben Person, die auch die griechische Brauerei Fix gründete."

# "Billy Mitchell (Videospieler)" via Wikipedia: https://de.wikipedia.org/w/index.php?title=Billy_Mitchell_(Videospieler)&oldid=262192596, CC BY-SA 4.0.
text2 = "William „Billy“ James Mitchell (* 16. Juli 1965 in Holyoke, Massachusetts)[1][2] wurde durch die Aufstellung von Rekorden in Arcade-Spielen bekannt.[3] Mitchell gilt als erster Spieler, der ein perfektes Pac-Man-Spiel gespielt hat. Er wurde u. a. als „[...] vermutlich größter Arcade-Videospiel-Spieler aller Zeiten [...]“ bezeichnet.[4] Nach Hinweisen auf Unregelmäßigkeiten in Aufzeichnungen seiner Rekorde wurden Mitchells Bestwerte 2018 wegen des Vorwurfs des Betrugs aus Rekordlisten von Twin Galaxies und Guinness World Records gestrichen.[5][6] Mitchell drohte infolge mit Klagen gegen beide Parteien wegen Verleumdung, worauf Guinness die Rekorde 2020 wieder anerkannte und Twin Galaxies gegenklagte.[7][8] Anfang 2024 einigten sich Billy Mitchell und Twin Galaxies gerichtlich, dass seine Rekorde durch Hardwarefehler theoretisch möglich gewesen seien, sodass diese zumindest in einer neuen historischen Sektion der Website wieder abrufbar sind.[9] Bis heute werden seine Rekorde von der Gaming-Community stark angezweifelt." 

#text3 = 

Zum Anzeigen des Inhalts einer Variable können wir die `print()`-Funktion nutzen:

In [2]:
print(text1)

Mets (griechisch Μετς) ist ein Stadtteil der griechischen Hauptstadt Athen. Im Viertel befanden sich früher zahlreiche Stadtvillen, von denen einige im nördlichen Teil bis heute erhalten sind. Auf dem Hügel Logginos wurde inzwischen ein Erholungspark angelegt. Mets verdankt seinen Namen einer Bierbrauerei, die vom bayerischen Brauer Karl Fuchs eröffnet wurde, derselben Person, die auch die griechische Brauerei Fix gründete.


In [10]:
print(text2)

William „Billy“ James Mitchell (* 16. Juli 1965 in Holyoke, Massachusetts)[1][2] wurde durch die Aufstellung von Rekorden in Arcade-Spielen bekannt.[3] Mitchell gilt als erster Spieler, der ein perfektes Pac-Man-Spiel gespielt hat. Er wurde u. a. als „[...] vermutlich größter Arcade-Videospiel-Spieler aller Zeiten [...]“ bezeichnet.[4] Nach Hinweisen auf Unregelmäßigkeiten in Aufzeichnungen seiner Rekorde wurden Mitchells Bestwerte 2018 wegen des Vorwurfs des Betrugs aus Rekordlisten von Twin Galaxies und Guinness World Records gestrichen.[5][6] Mitchell drohte infolge mit Klagen gegen beide Parteien wegen Verleumdung, worauf Guinness die Rekorde 2020 wieder anerkannte und Twin Galaxies gegenklagte.[7][8] Anfang 2024 einigten sich Billy Mitchell und Twin Galaxies gerichtlich, dass seine Rekorde durch Hardwarefehler theoretisch möglich gewesen seien, sodass diese zumindest in einer neuen historischen Sektion der Website wieder abrufbar sind.[9] Bis heute werden seine Rekorde von der Gam

## 2. Data Cleaning / Text Preprocessing

Bevor wir verschiedene Analysen am Text durchführen können, müssen zunächst Schritte zur Textbereinigung und -aufbereitung durchgeführt werden. Dazu können gehören:
* Entfernung von "noise": Text, Fußnoten, Satzzeichen, Tags, Symbole, Leerzeichen ...
* Entfernung von "stop words"
* Normalisierung:
  * Stemming
  * Lemmatisierung
  * lowercase
* Segmentierung
  * Satzsegmentierung
  * tokenization 

### 2.1 Noise Removal

Welche Schritte sind konkret anzuwenden? Dafür schauen wir uns den Text bzw. Textbeispiele an. `text1` weist bis auf ein griechisches Wort wenig Besonderheiten auf, in `text2` finden wir u.a. Auslassungspunkte (`[...]`) sowie die ursprünglichen Einzelnachweis-Verlinkung (z.B. `[1]`) des Wikipedia-Eintags.

Solche kleineren Relikte können wir gut mithilfe sogenannter regulärer Ausdrücke ("Regex") bereinigen und hierzu die Python-Bibliothek `re` nutzen. Wir nutzen die [`sub()`](https://docs.python.org/3/library/re.html#re.sub)-Funktion und ersetzen alle Einzelnachweise im Text mit leeren Strings (`""`) und ordnen dem aktualisierten Text eine neue Variable zu (`text2_cleaned`).

💡 **Aufgabe**: Entferne jetzt noch die Auslassungszeichen aus dem Text. Die [Dokumentation der re-Bibliothek](https://docs.python.org/3/library/re.html) oder die Seite [regex101](https://regex101.com/) können z.B. weiterhelfen.

<details>
    <summary><b>Lösung</b></summary>
    Eine Möglichkeit besteht darin, als Muster den regulären Ausdruck <code>\[.{3}\]</code> zu verwenden. Um <code>text2_cleaned</code> nochmal zu überarbeiten, können wir also unter dem bisherigen Code schreiben: 
    <ul>
        <li><code>text2_cleaned = re.sub(r"\[.{3}\]", "", text2_cleaned)</code></li>
</details>

In [18]:
text2_cleaned = re.sub(r"\[[0-9]\]", "", text2)
print(text2_cleaned)

William „Billy“ James Mitchell (* 16. Juli 1965 in Holyoke, Massachusetts) wurde durch die Aufstellung von Rekorden in Arcade-Spielen bekannt. Mitchell gilt als erster Spieler, der ein perfektes Pac-Man-Spiel gespielt hat. Er wurde u. a. als „[...] vermutlich größter Arcade-Videospiel-Spieler aller Zeiten [...]“ bezeichnet. Nach Hinweisen auf Unregelmäßigkeiten in Aufzeichnungen seiner Rekorde wurden Mitchells Bestwerte 2018 wegen des Vorwurfs des Betrugs aus Rekordlisten von Twin Galaxies und Guinness World Records gestrichen. Mitchell drohte infolge mit Klagen gegen beide Parteien wegen Verleumdung, worauf Guinness die Rekorde 2020 wieder anerkannte und Twin Galaxies gegenklagte. Anfang 2024 einigten sich Billy Mitchell und Twin Galaxies gerichtlich, dass seine Rekorde durch Hardwarefehler theoretisch möglich gewesen seien, sodass diese zumindest in einer neuen historischen Sektion der Website wieder abrufbar sind. Bis heute werden seine Rekorde von der Gaming-Community stark angezwe